![logo_ironhack_blue 7](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)


---


# **Lab-18: Ensembles.**


---

### **Alumno:** *Juan Alberto Peñalver Alvarez*


---

# Challenge 1

The heart disease dataset is a classic dataset that contains various health metrics (age, sex, chest pain type, blood pressure, cholesterol, etc.) related to diagnosing heart disease (binary classification: presence or absence of heart disease).

In [1]:
!pwd


/content


In [2]:
import os
print(os.getcwd())


/content


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the dataset (change the path if needed)
# df = pd.read_csv('../data/heart.csv')

In [5]:
# Importa cualquier biblioteca que puedas necesitar y los datos
from pathlib import Path

# Cambia a la carpeta que quieras
#%cd /content/drive/MyDrive/nombre_de_tu_carpeta
%cd /content/drive/MyDrive/IronHack/csv/

# Cambia a la carpeta que quieras
# %cd /content/drive/MyDrive/IronHack/Data_Machine_Learning/Lab&Projects/Lab-14/lab-regression-analysis-en-main_solved/lab-regression-en-main/your-code/
# vehicles_path ="/content/drive/MyDrive/IronHack/csv/"
#"/Data_Machine_Learning/Labs&Projects/Lab-14/lab-regression-analysis-en-main_solved/lab-regression-en-main/vehicles.csv"

# El cuaderno se encuentra en ./your-code; los archivos de datos suelen estar un nivel arriba
candidates = [
    Path('./data/heart.csv'),
    Path('../data/heart.csv'),
    Path('../../data/heart.csv'),
]

data_path = next((p for p in candidates if p.exists()), None)
print(data_path)

if data_path is None:
    raise FileNotFoundError('Could not find website.csv in expected locations. website.csv not found. Tried: ' + ', '.join(str(p) for p in candidate_paths))

# Load the dataset (change the path if needed)
df = pd.read_csv('./data/heart.csv', sep=',')
df.head()

/content/drive/MyDrive/IronHack/csv
data/heart.csv


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [6]:
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


We are going to try to predict the presence of hart disease suing this features, starting with a classical baseline method and trying to improve on that result with a series of ensembled approaches.

In [7]:
X = df.drop(columns="target")
y = df["target"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

# Feature scaling (for certain models, e.g., SVM or logistic regression, not always necessary for trees)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Baseline model : decision Tree

We'll train a decision tree as our baseline model and evaluate it using accuracy.

In [8]:
#Create and Train a Decision Tree Classifier and print the train and test accuracy

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Train Decision Tree
dt = DecisionTreeClassifier(random_state=0)
dt.fit(X_train, y_train)

# Predictions and evaluation
y_train_pred = dt.predict(X_train)
y_test_pred = dt.predict(X_test)

# Evaluate performance
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"Decision Tree -> Train accuracy: {train_acc:.4f} | Test accuracy: {test_acc:.4f}")


Decision Tree -> Train accuracy: 1.0000 | Test accuracy: 0.7895


We can see that this model is overfitting. This is expected, decision trees, especially deep ones  are notorious agressive at exploiting the data available. But that also makes them highly variant: a small change on the tree/data makes for potentially large changes in performance.

In [9]:
# Run the same code again a couple of times.
# You can see that the Train Accuracy is always 100% (overfitting) and the Test Accuracy is all over the place.
# This is undesirable: our method is not generalizing and has high variance

# Bagging: reducing variance

Bagging improves models because it reduces variance by averaging the predictions of multiple models trained on different subsets of the training data. This averaging effect reduces the sensitivity of the overall model to any one dataset or model, making the final prediction more stable and less prone to overfitting.

- High-variance models, like decision trees, tend to overfit the training data. This means that small changes in the training data can lead to large changes in the model’s predictions. For example, a decision tree trained on one subset of data might look completely different from a decision tree trained on another subset. This leads to high variance, where the model’s performance fluctuates a lot depending on the specific data it was trained on.
- Once all the individual models are trained, Bagging combines their predictions by averaging them (for regression) or using a majority vote (for classification). The key idea here is that the errors in each individual model are somewhat independent because they are trained on different bootstrap samples. Some models will make errors in one direction, while others might make errors in another. When you average these predictions, the errors cancel out, reducing the overall variability (variance) of the final model.

In [10]:
# Create and Train a BaggingClassifier.
# Use as base estimator a weak decision tree (max_depth=1) and 100 estimators to really over a lot of different data samples
# Print the train and test accuracy

from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Train BaggingClassifier
base_tree = DecisionTreeClassifier(max_depth=1, random_state=0)
bag = BaggingClassifier(
    estimator=base_tree,
    n_estimators=100,
    bootstrap=True,
    random_state=0,
    n_jobs=1
)
bag.fit(X_train, y_train)

# Predictions and evaluation
y_train_pred = bag.predict(X_train)
y_test_pred = bag.predict(X_test)

# Evaluate performance
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"Bagging (stumps) -> Train accuracy: {train_acc:.4f} | Test accuracy: {test_acc:.4f}")


Bagging (stumps) -> Train accuracy: 0.8194 | Test accuracy: 0.8421


You can probably see a modest improvement in score, but most importantly, the overfitting is mostly gone. This is because averaging over multiple datasets stabilizes the high variance of the base model.

In [11]:
# Run the same code again a couple of times.
# You can see that consistently the Train Accuracy is close to the Test Accuracy.

# Ejecute el mismo código un par de veces.
# Puede ver que, consistentemente, la precisión del entrenamiento se acerca a la precisión de la prueba.

# Boosting: reducing bias

Now we’ll apply AdaBoost with decision trees as weak learners. This will sequentially improve the model by focusing on difficult cases.

Boosting reduces bias by sequentially training a series of weak learners (often simple models like decision trees) where each subsequent model focuses on the mistakes made by the previous models. The key idea behind boosting is to incrementally improve the model by correcting errors, which helps to reduce bias, especially when the initial model is too simple and underfits the data.

- Boosting typically uses weak learners, which are models that perform only slightly better than random guessing. For example, in classification, a weak learner might be a shallow decision tree (a "stump") with just a few levels. Weak learners usually have high bias, meaning they are too simplistic and don't capture the underlying patterns in the data well. As a result, they underfit the data.

- In each iteration, boosting trains a new model that tries to correct the errors made by the earlier models. If an instance was misclassified by the first weak learner, it will receive a higher weight, so the next model pays more attention to it. As the sequence of models progresses, the ensemble collectively focuses more on the difficult-to-predict instances. Over time, the combined models become better at fitting the data, as they successively reduce the bias (systematic error) by adjusting for earlier mistakes.

In [12]:
# Create and Train a AdaBoostClassifier.
# Use as base estimator a weak decision tree (max_depth=1) and 100 estimators to really target the specific behaviors of this phenomenon
# Print the train and test accuracy

from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Train AdaBoost
base_tree = DecisionTreeClassifier(max_depth=1, random_state=0)
ada = AdaBoostClassifier(
    estimator=base_tree,
    n_estimators=100,
    learning_rate=1.0,
    random_state=0,
    algorithm="SAMME"
)
ada.fit(X_train, y_train)

# Predictions and evaluation
y_train_pred = ada.predict(X_train)
y_test_pred = ada.predict(X_test)

# Evaluate performance
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"AdaBoost (stumps) -> Train accuracy: {train_acc:.4f} | Test accuracy: {test_acc:.4f}")


/usr/local/lib/python3.12/dist-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(


AdaBoost (stumps) -> Train accuracy: 0.9119 | Test accuracy: 0.8684


You can probably see a good improvement in score, but overfitting rearing it's ugly head a gain (not as much as in the base model). This is because the iterative correction of adaboost really allows the model to focus on the specifics of this problem, at a cost of overexploiting the dataset.

In [14]:
# Run the same code again a couple of times.
# You can see that the test Accuracy will mostly be pretty good, even if some times it get's lower or higher scores (high variance, low bias)
# You can see also that consistently the Train Accuracy is higher than the Test Accuracy,indicating some (not extreme) overfitting

# Ejecute el mismo código un par de veces.
# Puede observar que la precisión de la prueba será bastante buena en general, incluso si a veces obtiene puntuaciones más bajas o más altas (alta varianza, bajo sesgo).
# También puede observar que, consistentemente, la precisión del entrenamiento es mayor que la precisión de la prueba, lo que indica un sobreajuste (no extremo).